# residual-skip-add — worked example 2: Residual block with downsampling projection shortcut

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `residual-skip-add`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the residual branch changes channels or spatial resolution (stride > 1), the identity shortcut can no longer be added because shapes mismatch. The fix is a projection shortcut: a 1x1 convolution with the same stride that matches the residual branch's output shape, so `F(x) + shortcut(x)` is well-defined.

## Worked solution

Here the residual function downsamples with a stride-2 3x3 conv and lifts channels from 16 to 32, producing a `(B, 32, H/2, W/2)` output. The input `x` is `(B, 16, H, W)`, so a plain identity add would fail. We add a 1x1 conv shortcut with stride 2 and 32 out-channels, which produces a tensor of exactly the residual branch's shape. Forward sums them. We run a `(2, 16, 32, 32)` input and verify the output is `(2, 32, 16, 16)` — both branches agree on shape because the projection matches the downsample.

In [ ]:
import torch.nn as nn
Tensor = t.Tensor


class DownResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.f = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0)

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
block = DownResBlock(16, 32, stride=2)
x = t.randn(2, 16, 32, 32)
out = block(x)
print('output shape:', tuple(out.shape))
print('branches match:', block.f(x).shape == block.skip(x).shape)